# Interactive CNMF-E (1p calcium imaging)

Step-by-step version of the `caiman-cnmfe` CLI for **one** movie, with inspection and parameter tuning between steps.
It calls the same `caiman_tools.pipeline` functions as the CLI, so results are identical for identical parameters.

**Workflow**
1. Configure movie + (optional) params JSON
2. Inspect raw movie → motion correction
3. Tune `gSig`, `min_corr`, `min_pnr` on the correlation / PNR images
4. Fit CNMF-E
5. Tune `min_SNR`, `rval_thr` (component evaluation, re-runnable without refitting)
6. dF/F + deconvolution, inspect components
7. Save `<movie>_caiman.hdf5`, the viewer HTML, and the **tuned params JSON** → reuse for batch runs:
   `caiman-cnmfe *.tif --params <movie>_caiman_params.json`

All tunable parameters live in the `params` dict (same layout as the CLI's params JSON); `set_params(...)` updates it *and* the CaImAn parameter object together.

## 1. Setup

In [ ]:
import os

# Must be set before numpy / CaImAn spin up their thread pools
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

from pathlib import Path

import bokeh.plotting as bpl
import cv2
import holoviews as hv
import matplotlib.pyplot as plt
import numpy as np
import psutil
import tifffile

import caiman as cm
from caiman.utils.visualization import nb_inspect_correlation_pnr, view_quilt

from caiman_tools import __version__
from caiman_tools import pipeline as pl
from caiman_tools.logging_utils import setup_logging
from caiman_tools.params import load_params, save_params
from caiman_tools.view import estimates_view_layout, render_estimates_view

try:
    cv2.setNumThreads(0)
except Exception:
    pass

%matplotlib widget
bpl.output_notebook()
hv.notebook_extension("bokeh")

print(f"caiman_tools {__version__}, CaImAn {cm.__version__}, {psutil.cpu_count()} CPUs")

## 2. Configuration

In [ ]:
MOVIE_PATH = Path(
    "/nfs/scistore19/jonasgrp/sbhattac/MiniCA3_pilot/Outputs_pilot/SB_PJ_20260624-154509/SB_PJ_2026-06-24-15-57-23_video_trig_0_DeinterleavedPlane_580_multiplanar_nPP.tiff"
)
PARAMS_JSON = None        # e.g. Path("my_params.json"); None -> package defaults
FR = None                 # frame rate (Hz); None -> from <stem>_timestamps.csv next to the movie
N_PROCESSES = 4          # None -> params["pipeline"]["n_processes"] / CaImAn default
CAIMAN_LOG_LEVEL = "NONE" # "NONE", "WARNING", "INFO", "DEBUG": CaImAn's internal logging

OUTPUT_PATH = pl.output_path_for(MOVIE_PATH)                          # <stem>_caiman.hdf5
VIEW_HTML = OUTPUT_PATH.with_name(f"{OUTPUT_PATH.stem}_view.html")
PARAMS_OUT = OUTPUT_PATH.with_name(f"{OUTPUT_PATH.stem}_params.json")

setup_logging(CAIMAN_LOG_LEVEL)
assert MOVIE_PATH.exists(), MOVIE_PATH
if OUTPUT_PATH.exists():
    print(f"NOTE: {OUTPUT_PATH.name} already exists and will be overwritten when saving")

In [ ]:
params = load_params(PARAMS_JSON)   # {"cnmf_params": {...}, "pipeline": {...}}
parameters, MOTION_CORRECT = pl.build_cnmf_params(MOVIE_PATH, params["cnmf_params"])


def set_params(**kwargs):
    # Update the params dict (exported at the end) and the CaImAn CNMFParams object together.
    # After the model is created, model.params is this same object.
    params["cnmf_params"].update(kwargs)
    parameters.change_params(params_dict={k: v for k, v in kwargs.items() if k != "motion_correct"})


# Per-plane frame rate: FR > <stem>_timestamps.csv > params
set_params(fr=pl.resolve_frame_rate(MOVIE_PATH, params["cnmf_params"]["fr"], FR))

params["cnmf_params"]

## 3. Raw movie

Check frame count, dimensions and that `fr` matches the acquisition frame rate.

In [ ]:
movie_orig = cm.load(str(MOVIE_PATH))
T, d1, d2 = movie_orig.shape
print(f"{MOVIE_PATH.name}: {T} frames, {d1}x{d2} px, dtype={movie_orig.dtype}, "
      f"fr={params['cnmf_params']['fr']} Hz -> {T / params['cnmf_params']['fr']:.1f} s")

fig, axs = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
for ax, (name, img) in zip(axs, [("std", movie_orig.std(0)), ("max", movie_orig.max(0))]):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{name} projection")
    ax.axis("off")

In [ ]:
# Optional: scroll through the raw movie
tifffile.imshow(np.asarray(movie_orig), cmap="gray");

## 4. Start cluster

In [ ]:
if "cluster" in globals():
    cm.stop_server(dview=cluster)
_, cluster, n_processes = cm.cluster.setup_cluster(
    backend="multiprocessing",
    n_processes=N_PROCESSES or params["pipeline"].get("n_processes"),
    ignore_preexisting=False,
)
print(f"Cluster with {n_processes} processes")

## 5. Motion correction

Relevant params: `pw_rigid`, `gSig_filt`, `max_shifts`, `strides`, `overlaps`, `max_deviation_rigid`, `border_nan`.
Change with e.g. `set_params(pw_rigid=True)` and re-run this cell.

In [ ]:
%%time
mc = pl.motion_correct_movie(MOVIE_PATH, parameters, cluster, MOTION_CORRECT)
images = pl.load_images(mc.fname_mmap)   # (T, d1, d2) memmap, input to CNMF-E
print(f"memmap: {Path(mc.fname_mmap).name}, border_px={mc.border_px}")

In [ ]:
if mc.mot_correct is not None:
    m = mc.mot_correct
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True,
                            gridspec_kw=dict(width_ratios=[2, 1, 1]))
    if m.pw_rigid:
        axs[0].plot(np.asarray(m.x_shifts_els), lw=0.5)
        axs[0].set_title("pw-rigid x shifts (per patch)")
    else:
        axs[0].plot(np.asarray(m.shifts_rig))
        axs[0].legend(["x", "y"])
        axs[0].set_title("rigid shifts")
    axs[0].set(xlabel="frame", ylabel="px")
    for ax, (name, img) in zip(axs[1:], [("raw mean", movie_orig.mean(0)),
                                         ("corrected mean", images.mean(0))]):
        ax.imshow(img, cmap="gray")
        ax.set_title(name)
        ax.axis("off")

## 6. Initialization parameters: correlation / PNR

CNMF-E seeds neurons at pixels where **both** correlation ≥ `min_corr` and PNR ≥ `min_pnr`.
Use the interactive plot to find thresholds that keep neuron centres but drop background/vessels.

* `gSig`: expected neuron half-width (px), also the filter size for these images. Too large blends neighbouring neurons.
  If you change it, re-run the cell below, and set `gSiz = 2*gSig + 1`.

In [ ]:
set_params(gSig=[3, 3], gSiz=[7, 7])   # change & re-run to compare

correlation_image, pnr_image = pl.correlation_pnr(images, params["cnmf_params"]["gSig"][0])
nb_inspect_correlation_pnr(correlation_image, pnr_image, cmap="inferno")

In [ ]:
set_params(min_corr=0.85, min_pnr=12)

fig, axs = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
seeds = (correlation_image >= params["cnmf_params"]["min_corr"]) & (pnr_image >= params["cnmf_params"]["min_pnr"])
for ax, (name, img) in zip(axs, [("correlation", correlation_image), ("PNR", pnr_image),
                                 ("candidate seed pixels", seeds)]):
    ax.imshow(img, cmap="inferno")
    ax.set_title(name)
    ax.axis("off")
print(f"{seeds.sum()} pixels pass both thresholds")

### Patches

`rf` = patch half-width, `stride` = overlap (px). Patches should hold several neuron diameters; larger patches = less parallelism.

In [ ]:
set_params(rf=40, stride=20)

patch_width = 2 * params["cnmf_params"]["rf"] + 1
patch_overlap = params["cnmf_params"]["stride"] + 1
ax = view_quilt(correlation_image, patch_width - patch_overlap, patch_overlap,
                vmin=np.percentile(correlation_image, 50), vmax=np.percentile(correlation_image, 99.5),
                color="yellow", figsize=(4, 4))
ax.set_title(f"patch width {patch_width}, overlap {patch_overlap}");

## 7. Fit CNMF-E

The slow step. Other relevant params: `merge_thr`, `tsub`/`ssub` (downsampling for init; increase if memory is short), `ssub_B`, `ring_size_factor`, `p`.

In [ ]:
%%time
model = pl.fit_cnmfe(images, parameters, cluster, n_processes, mc.border_px)
print(f"{len(model.estimates.C)} components found")

## 8. Component evaluation

Components are accepted if they pass **SNR** (`min_SNR`) **or** spatial correlation (`rval_thr`) criteria
(and are not rejected by the lower hard limits). Re-run these two cells to retune — no refit needed.

In [ ]:
set_params(min_SNR=3, rval_thr=0.85, use_cnn=False)

pl.evaluate_components(model, images, cluster)
n_total, n_accepted, n_rejected, snr_accepted = pl.summarize_components(model, MOVIE_PATH.name)

In [ ]:
est = model.estimates
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
for idx, label, color in [(est.idx_components, "accepted", "tab:green"),
                          (est.idx_components_bad, "rejected", "tab:red")]:
    ax.scatter(est.r_values[idx], est.SNR_comp[idx], s=10, c=color, label=f"{label} ({len(idx)})")
ax.axhline(params["cnmf_params"]["min_SNR"], c="k", ls="--", lw=0.8)
ax.axvline(params["cnmf_params"]["rval_thr"], c="k", ls="--", lw=0.8)
ax.set(xlabel="spatial correlation (r_value)", ylabel="SNR", yscale="log")
ax.legend();

In [ ]:
est.plot_contours_nb(img=correlation_image, idx=est.idx_components);

## 9. dF/F and deconvolution

Detrended dF/F (`F_dff`), then deconvolution of it (`S_dff`), as in the CLI. Params: `params["pipeline"]["detrend_df_f"]`.

In [ ]:
params["pipeline"]["detrend_df_f"].update(quantileMin=8, frames_window=250, flag_auto=False, use_residuals=False)

model.estimates.F_dff = None   # force recompute if this cell is re-run
pl.compute_dff_and_deconvolve(model, params["pipeline"]["detrend_df_f"])
model.estimates.Cn = correlation_image

## 10. Inspect components

Same viewer as the `_caiman_view.html` written by the CLI (click legend entries to hide traces).

In [ ]:
bpl.show(estimates_view_layout(model.estimates, which="accepted"))

In [ ]:
# Rejected components - check nothing good was thrown away
if len(model.estimates.idx_components_bad):
    bpl.show(estimates_view_layout(model.estimates, which="rejected"))

## 11. Save

Writes the CaImAn hdf5 (as `caiman-cnmfe` does), the standalone viewer HTML, and the tuned params as JSON.

In [ ]:
pl.save_model(model, OUTPUT_PATH, correlation_image)
if n_accepted := len(model.estimates.idx_components):
    render_estimates_view(model.estimates, which="accepted", output_html=VIEW_HTML,
                          title=f"{OUTPUT_PATH.name} (accepted)")
save_params(params, PARAMS_OUT)

print(f"{MOVIE_PATH.name}: {n_accepted} accepted / {len(model.estimates.idx_components_bad)} rejected"
      f" -> {OUTPUT_PATH}")
print(f"Batch-process more movies with these params:\n  caiman-cnmfe <tifs> --params {PARAMS_OUT}")

### Reload a saved result

`load_CNMF` restores the model (with `Cn`); the memmap is only needed for steps that touch the movie (evaluation, background).

In [ ]:
from caiman.source_extraction.cnmf.cnmf import load_CNMF

# model = load_CNMF(str(OUTPUT_PATH), n_processes=n_processes, dview=cluster)
# correlation_image = model.estimates.Cn

## 12. Clean up

In [ ]:
DELETE_MEMMAP = False   # remove intermediate .mmap files (large!)

cm.stop_server(dview=cluster)
del cluster
if DELETE_MEMMAP:
    del images
    pl.cleanup_memmap_files(mc)